In [1]:
# Library yang digunakan

import pandas as pd  # untuk manipulasi dan analisis data
import numpy as np   # untuk operasi numerik
import re            # untuk ekspresi reguler pada cleaning teks

# **DATA WRANGLING**

## **Gathering Data**
Tahap ini difokuskan pada pengumpulan data mentah berupa kumpulan judul berita ekonomi dan pergerakan saham dari portal CNBC Indonesia yang berfungsi sebagai fondaki utama pengembangan fitur analisis sentimen pada platform website. Data yang dikumpulkan ini diproses sebagai dataset pelatihan guna melatih model AI dalam mengklasifikasikan sentimen pasar secara otomatis ke dalam kategori positif, negatif, atau netral. 

In [2]:
beritasaham_df = pd.read_csv('Dataset-CNBCI-Sentimented.csv')

In [3]:
beritasaham_df.shape  # untuk mengecek jumlah baris dan kolom

(9819, 3)

**Hasil:**
- dataset berita saham memiliki 9819 baris
- dataset berita saham memiliki 3 kolom

Hal ini menunjukkan bahwa dataset berisi cukup banyak data berita saham yang siap digunakan untuk proses analisis lebih lanjut, seperti data cleaning, exploratory data analysis (EDA), maupun analisis sentimen.


In [4]:
beritasaham_df.head()   # untuk menampilkan 5 baris utama

,judul,tanggal,sentimen
0,"Direktur Garuda Salman El Farisiy Meninggal, I...",2024/01/01,negatif
1,"Catatan Sejarah 2023, Indonesia Luncurkan Burs...",2024/01/01,netral
2,"Ramalan Bitcoin Paling Gila 2024, Naik 1.000% ...",2024/01/01,netral
3,Wah! Segini Harta Mantan Istri Prabowo Titiek ...,2024/01/01,netral
4,"Pensiun di Oktober 2024, Jokowi Kantongi Uang ...",2024/01/01,netral


**Hasil:**
- Berdasarkan 5 data teratas, dataset memiliki tiga kolom utama yaitu `judul`, `tanggal`, dan `sentimen`. Data berisi berita dengan label sentimen seperti negatif dan netral.
- Dari sampel awal terlihat bahwa sentimen netral lebih mendominasi dibanding sentimen lainnya, sehingga distribusi data kemungkinan tidak seimbang. Selain itu, beberapa berita tidak berhubungan langsung dengan saham atau perusahaan tertentu, sehingga terdapat noise pada dataset.
- Dataset ini juga belum memiliki informasi kode saham, sehingga analisis lebih berfokus pada sentimen berita pasar secara umum, bukan pergerakan saham perusahaan tertentu.

## **Assessing Data**
Tahap assessing data dilakukan untuk mengevaluasi struktur, kualitas, dan kondisi awal dataset sebelum memasuki proses data cleaning. Pada tahap ini dilakukan pengecekan tipe data pada setiap kolom, melihat jumlah dan distribusi label sentimen, mendeteksi data duplikat, serta memeriksa adanya missing value atau nilai kosong. Selain itu, dilakukan juga pengecekan isi data secara umum untuk memastikan data relevan dan sesuai dengan kebutuhan analisis. Hasil dari proses ini digunakan sebagai dasar dalam menentukan langkah pembersihan dan pengolahan data selanjutnya.

In [5]:
beritasaham_df.dtypes   # untuk cek tipe data

judul       str
tanggal     str
sentimen    str
dtype: object

**Hasil:**

Berdasarkan hasil pengecekan, seluruh kolom masih memiliki tipe data `object`. Kolom `tanggal` perlu diubah ke format `datetime` agar dapat digunakan untuk analisis berbasis waktu, sedangkan kolom `sentimen` dapat diubah ke bentuk numerik (encoding) untuk mendukung proses analisis dan pemodelan lebih lanjut.

In [6]:
beritasaham_df.isnull().sum()   # untuk cek missing values

judul       0
tanggal     0
sentimen    0
dtype: int64

**Hasil:**

Hasil pengecekan menunjukkan tidak terdapat missing value pada seluruh kolom (`judul`, `tanggal`, dan `sentimen`). Hal ini menandakan data sudah lengkap sehingga tidak memerlukan penanganan nilai kosong sebelum analisis dilakukan.

In [7]:
print('Jumlah duplikat:', beritasaham_df.duplicated().sum())   # untuk cek jumlah baris duplikat

Jumlah duplikat: 3


**Hasil:**

Ditemukan sebanyak 3 data duplikat pada dataset. Data duplikat perlu dihapus agar tidak memengaruhi hasil analisis dan menjaga kualitas data tetap baik.

In [8]:
# untuk cek distribusi label sentimen

beritasaham_df['sentimen'].value_counts()
beritasaham_df['sentimen'].value_counts(normalize=True).round(2)

sentimen
netral     0.44
positif    0.29
negatif    0.26
Name: proportion, dtype: float64

**Hasil:**

Distribusi label sentimen menunjukkan data tidak sepenuhnya seimbang. Label `netral` memiliki proporsi paling besar sebesar 44% (0.44), sedangkan label `positif` sebesar 29% (0.29) dan `negatif` sebesar 26% (0.26). Kondisi ini menunjukkan sentimen netral lebih dominan dalam dataset.

In [9]:
# cek sampel judul untuk identifikasi karakter yang perlu dibersihin

print(beritasaham_df['judul'].sample(5, random_state=42).to_string())

9533    Ketua Komisi XI DPR Ungkap Pentingnya Pasar Mo...
3755    Emiten Raam Punjabi (RAAM) Private Placement 6...
647     Cek Saldo Minimum BCA, Mandiri, BNI, BRI Terbaru!
9525    Ketua Komisi XI DPR: Revisi UU P2SK untuk Perk...
360     Angin Segar Kepemimpinan UNVR, Mengenal Benjie...


**Hasil:**

Beberapa judul berita masih mengandung karakter `\n` serta spasi berlebih, sehingga perlu dilakukan pembersihan agar teks lebih rapi dan konsisten untuk proses analisis selanjutnya.

## **Cleaning Data**

Berdasarkan hasil proses assessing data, ditemukan beberapa hal yang perlu diperbaiki sebelum data digunakan untuk analisis lebih lanjut, yaitu:

1. Terdapat 3 baris data duplikat yang perlu dihapus agar tidak memengaruhi hasil analisis.
2. Kolom `tanggal` masih bertipe `object`, sehingga perlu dikonversi ke format `datetime` agar dapat digunakan untuk analisis berbasis waktu.
3. Kolom `judul` masih mengandung karakter `\n` dan spasi berlebih, sehingga perlu dibersihkan agar teks lebih rapi dan konsisten.
4. Kolom `sentimen` perlu diubah ke bentuk numerik (encoding) agar dapat diproses pada tahap analisis dan pemodelan data.

In [10]:
# buat salinan dataframe agar dataframe asli tidak berubah

berita_df = beritasaham_df.copy()

In [11]:
# hapus baris duplikat

berita_df = berita_df.drop_duplicates()
print("Jumlah baris setelah hapus duplikat:", berita_df.shape[0])

Jumlah baris setelah hapus duplikat: 9816


In [12]:
# konversi kolom tanggal dari object ke datetime
berita_df['tanggal'] = pd.to_datetime(berita_df['tanggal'])

berita_df['tanggal'].dtype
berita_df['tanggal'].min(), "->", berita_df['tanggal'].max()

(Timestamp('2024-01-01 00:00:00'), '->', Timestamp('2025-03-31 00:00:00'))

In [13]:
# hapus karakter \n dan \t
berita_df['judul'] = berita_df['judul'].str.replace('\n', ' ', regex=False)
berita_df['judul'] = berita_df['judul'].str.replace('\t', ' ', regex=False)

# hapus spasi di awal dan akhir
berita_df['judul'] = berita_df['judul'].str.strip()

# normalisasi spasi berlebih menjadi satu spasi
berita_df['judul'] = berita_df['judul'].str.replace(r'\s+', ' ', regex=True)

In [14]:
# cek hasil cleaning

print(berita_df['judul'].sample(5, random_state=42).to_string())

7720           Intip Strategi Investasi Ala Nabi Muhammad
3896    Jangan Asal Ngutang, Ini Daftar 98 Pinjol Beri...
9317    Ekonom Ungkap Efek Penurunan Daya Beli di Bali...
3595    Geber Tabungan Haji, Bank Mega Syariah Berangk...
5840    BSI Hadirkan Literasi Digital & Keuangan di Ma...


In [15]:
# encode label sentimen ke angka untuk keperluan training model
label_map = {'negatif': 0, 'netral': 1, 'positif': 2}
berita_df['sentimen_encoded'] = berita_df['sentimen'].map(label_map)

berita_df[['sentimen', 'sentimen_encoded']].value_counts()

sentimen  sentimen_encoded
netral    1                   4354
positif   2                   2887
negatif   0                   2575
Name: count, dtype: int64

In [16]:
# cek hasil akhir dataset setelah cleaning
berita_df.shape
berita_df.dtypes
berita_df.head()

,judul,tanggal,sentimen,sentimen_encoded
0,"Direktur Garuda Salman El Farisiy Meninggal, I...",2024-01-01,negatif,0
1,"Catatan Sejarah 2023, Indonesia Luncurkan Burs...",2024-01-01,netral,1
2,"Ramalan Bitcoin Paling Gila 2024, Naik 1.000% ...",2024-01-01,netral,1
3,Wah! Segini Harta Mantan Istri Prabowo Titiek ...,2024-01-01,netral,1
4,"Pensiun di Oktober 2024, Jokowi Kantongi Uang ...",2024-01-01,netral,1


In [17]:
berita_df.to_csv('dataset_berita.csv', index=False)

### **Kesimpulan:**

- Setelah proses cleaning selesai, dataset yang digunakan berisi 9.816 baris data berita saham dan keuangan yang mencakup rentang waktu dari 1 Januari 2024 hingga 31 Maret 2025.
- Proses cleaning yang dilakukan meliputi penghapusan baris duplikat, konversi kolom tanggal dari tipe object ke datetime, pembersihan teks judul dari karakter tidak perlu seperti \n, \t, dan spasi berlebih, serta normalisasi spasi ganda. Selain itu, label sentimen juga diencode ke angka yaitu negatif menjadi 0, netral menjadi 1, dan positif menjadi 2 agar siap digunakan untuk training model.
- Dari sisi distribusi sentimen, kategori netral mendominasi dengan 4.354 data (sekitar 44%), diikuti positif sebanyak 2.887 data (sekitar 29%), dan negatif sebanyak 2.575 data (sekitar 26%). Dominasi sentimen netral cukup wajar mengingat berita keuangan umumnya bersifat informatif. Sementara itu, proporsi sentimen positif dan negatif yang cukup seimbang menjadi nilai plus karena model tidak akan cenderung bias ke salah satu kelas.

Secara keseluruhan, dataset sudah bersih, terstruktur, dan siap digunakan untuk tahap selanjutnya yaitu pelatihan model klasifikasi sentimen. Hasil akhir dataset juga telah diekspor ke file dataset_berita.csv.

# **Conclusion**

Dataset berita saham telah berhasil melalui proses pembersihan data dan siap digunakan untuk tahap analisis maupun training model klasifikasi sentimen oleh AI Engineer. Berikut ringkasan perubahan yang dilakukan:

1. Sebanyak 3 baris data duplikat berhasil dihapus, sehingga jumlah data berubah dari 9819 menjadi 9816 baris.
2. Kolom `tanggal` berhasil dikonversi dari tipe `object` menjadi `datetime` agar dapat digunakan untuk analisis berbasis waktu.
3. Kolom `judul` telah dibersihkan dari karakter `\n`, `\t`, serta spasi berlebih sehingga teks menjadi lebih rapi dan konsisten.
4. Kolom `sentimen` berhasil diubah ke bentuk numerik (encoding), yaitu `negatif = 0`, `netral = 1`, dan `positif = 2`, sehingga siap digunakan dalam proses pemodelan machine learning.